# 01 · Download and explore GSE98793

**Aim:** download the GSE98793 dataset from GEO and check that the sample information matches the original publication before any gene expression analysis.

**Dataset:** whole-blood microarray (Affymetrix U133 Plus 2.0), 128 patients with major depressive disorder and 64 healthy controls ([Leday et al., 2018](https://pmc.ncbi.nlm.nih.gov/articles/PMC5720346)).

**Role in the project:** first independent dataset for testing the 16-gene panel from the BIODEP study.

## 1. Setup
Load the packages and check that the notebook runs in the project environment (`idt`).

In [1]:
import pandas as pd
import GEOparse

print("pandas", pd.__version__)

pandas 3.1.1


## 2. Download the data
Download GSE98793 from GEO into `data/raw/`. Raw data are not committed to GitHub: running this cell downloads them again.

In [ ]:
from pathlib import Path
import warnings

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.DtypeWarning)
    gse = GEOparse.get_GEO(geo="GSE98793", destdir=str(raw_dir), silent=True)

/Users/claudia/miniforge3/envs/idt/lib/python3.12/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


  ## 3. Sample information
  Each sample (one per participant) comes with clinical information. The relevant fields are those starting with `characteristics_ch1`.

In [3]:
pheno = gse.phenotype_data
print(pheno.shape)
pheno.columns.tolist()

(192, 37)


['title',
 'geo_accession',
 'status',
 'submission_date',
 'last_update_date',
 'type',
 'channel_count',
 'source_name_ch1',
 'organism_ch1',
 'taxid_ch1',
 'characteristics_ch1.0.subject group',
 'characteristics_ch1.1.anxiety',
 'characteristics_ch1.2.gender',
 'characteristics_ch1.3.age',
 'characteristics_ch1.4.tissue',
 'characteristics_ch1.5.batch',
 'molecule_ch1',
 'extract_protocol_ch1',
 'label_ch1',
 'label_protocol_ch1',
 'hyb_protocol',
 'scan_protocol',
 'description',
 'data_processing',
 'platform_id',
 'contact_name',
 'contact_email',
 'contact_laboratory',
 'contact_department',
 'contact_institute',
 'contact_address',
 'contact_city',
 'contact_zip/postal_code',
 'contact_country',
 'supplementary_file',
 'series_id',
 'data_row_count']

Keep the six relevant fields and give them short names.



In [4]:
columns = {
    "characteristics_ch1.0.subject group": "group",
    "characteristics_ch1.1.anxiety": "anxiety",
    "characteristics_ch1.2.gender": "sex",
    "characteristics_ch1.3.age": "age",
    "characteristics_ch1.4.tissue": "tissue",
    "characteristics_ch1.5.batch": "batch",
}

samples = pheno[list(columns)].rename(columns=columns)
samples.head()

,group,anxiety,sex,age,tissue,batch
GSM2612096,CNTL; healthy control,no,M,35.8,whole blood,1
GSM2612097,CNTL; healthy control,no,F,36.9,whole blood,1
GSM2612098,CNTL; healthy control,no,F,62,whole blood,1
GSM2612099,CNTL; healthy control,no,F,35.7,whole blood,1
GSM2612100,CNTL; healthy control,no,M,53.4,whole blood,1


## 4. Quality checks
Check that group sizes, sex, tissue and batch match the original publication.

In [5]:
for column in ["group", "anxiety", "sex", "tissue", "batch"]:
    print(samples[column].value_counts(), "\n")

pd.crosstab(samples["batch"], samples["group"])


group
CASE; major depressive disorder (MDD) patient    128
CNTL; healthy control                             64
Name: count, dtype: int64 

anxiety
no     128
yes     64
Name: count, dtype: int64 

sex
F    144
M     48
Name: count, dtype: int64 

tissue
whole blood    192
Name: count, dtype: int64 

batch
1    96
2    96
Name: count, dtype: int64 



group,CASE; major depressive disorder (MDD) patient,CNTL; healthy control
batch,,
1,64,32
2,64,32


Check anxiety and age by group.

In [6]:
print(pd.crosstab(samples["anxiety"], samples["group"]), "\n")

samples["age"] = pd.to_numeric(samples["age"])
samples.groupby("group")["age"].describe()

group    CASE; major depressive disorder (MDD) patient  CNTL; healthy control
anxiety                                                                      
no                                                  64                     64
yes                                                 64                      0 



,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
CASE; major depressive disorder (MDD) patient,128.0,52.039844,11.507049,31.0,43.775,52.65,62.05,72.8
CNTL; healthy control,64.0,52.029688,11.410420,31.2,44.050,52.55,62.05,73.1


  ## 5. Save the sample table
  Save the cleaned sample information to `data/processed/` for the next notebooks.

In [7]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

samples.to_csv(processed_dir / "GSE98793_samples.csv")

## Summary
- 192 whole-blood samples: 128 patients with major depressive disorder (64 with and 64 without anxiety) and 64 healthy controls.
- Sex ratio about 3:1 (144 female, 48 male).
- Age is matched between groups (mean 52.0 years in both).
- The two processing batches have the same case–control ratio (64 cases and 32 controls each), so batch is not confounded with diagnosis.